# Partie 3 - Spark SQL et DataFrames
Refaire et enrichir l'analyse de la Partie 2 avec l'API structuree (schemas explicites, SQL,
jointure, UDF, Parquet partitionne, plans Catalyst).

In [ ]:
import os, sys, json, statistics
sys.path.insert(0, "/home/sahlpay/jobs"); sys.path.insert(0, "../jobs")
from sahlpay_common import get_spark, Paths, LOG_DIR, TX_SCHEMA, MERCHANT_SCHEMA, merchants_path, timer
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

spark = get_spark("SahlPay-P3-SQL", cores_max=4)
sc = spark.sparkContext
timings = {}
os.makedirs(LOG_DIR, exist_ok=True)

## 1. Lecture avec schema explicite (`StructType`, sans inference)

In [ ]:
history_file = f"{Paths.bronze_history}/transactions_history.jsonl"
tx = (spark.read.schema(TX_SCHEMA)
      .option("mode", "PERMISSIVE")
      .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ssX")
      .json(history_file))
merchants = spark.read.schema(MERCHANT_SCHEMA).json(merchants_path())
tx.printSchema()
merchants.printSchema()
with timer("p3_count_events", timings):
    print("evenements :", tx.count(), "| merchants :", merchants.count())

## 2. Montant total (et moyen) par ville : API DataFrame puis SQL
Meme regle qu'en Partie 2 : transactions `COMPLETED`, dedoublonnees par `tx_id`.

In [ ]:
completed = tx.filter(F.col("status") == "COMPLETED").dropDuplicates(["tx_id"])

with timer("p3_total_by_city_df", timings):
    by_city_df = (completed.groupBy("city")
                  .agg(F.round(F.sum("amount_mad"), 2).alias("montant_total_mad"),
                       F.count("*").alias("nb_tx"),
                       F.round(F.avg("amount_mad"), 2).alias("montant_moyen_mad"))
                  .orderBy(F.desc("montant_total_mad")))
    by_city_df.show(truncate=False)

In [ ]:
tx.createOrReplaceTempView("transactions")
merchants.createOrReplaceTempView("merchants")

with timer("p3_total_by_city_sql", timings):
    by_city_sql = spark.sql("""
        WITH completed AS (
            SELECT tx_id, first(city) AS city, first(amount_mad) AS amount_mad
            FROM transactions
            WHERE status = 'COMPLETED'
            GROUP BY tx_id
        )
        SELECT city,
               round(sum(amount_mad), 2) AS montant_total_mad,
               count(*)                  AS nb_tx,
               round(avg(amount_mad), 2) AS montant_moyen_mad
        FROM completed
        GROUP BY city
        ORDER BY montant_total_mad DESC
    """)
    by_city_sql.show(truncate=False)

# Les deux resultats sont identiques
assert sorted(map(tuple, by_city_df.collect())) == sorted(map(tuple, by_city_sql.collect()))
print("DataFrame et SQL donnent le meme resultat")

## 3. Jointure P2M x merchants : montant encaisse par categorie et par ville

In [ ]:
p2m = completed.filter(F.col("type") == "P2M")
joined = p2m.join(merchants, p2m.payee_id == merchants.merchant_id, "inner")

with timer("p3_join_category_city", timings):
    by_cat_city = (joined.groupBy(merchants.category, merchants.city)
                   .agg(F.round(F.sum("amount_mad"), 2).alias("montant_total_mad"),
                        F.count("*").alias("nb_paiements"))
                   .orderBy(F.desc("montant_total_mad")))
    by_cat_city.show(20, truncate=False)

(joined.groupBy(merchants.category)
       .agg(F.round(F.sum("amount_mad"), 2).alias("montant_total_mad"), F.count("*").alias("nb"))
       .orderBy(F.desc("montant_total_mad"))
       .show(truncate=False))

## 4. UDF `amount_band` vs `when/otherwise`

In [ ]:
def amount_band(amount_mad):
    if amount_mad is None:
        return None
    if amount_mad < 50:
        return "micro"
    if amount_mad <= 2000:
        return "standard"
    return "eleve"

amount_band_udf = F.udf(amount_band, StringType())
spark.udf.register("amount_band", amount_band, StringType())   # utilisable aussi en SQL

def amount_band_native(col):
    return (F.when(col < 50, "micro")
             .when(col <= 2000, "standard")
             .otherwise("eleve"))

def bench(df_fn, label, runs=3):
    times = []
    import time
    for _ in range(runs):
        t0 = time.perf_counter()
        df_fn().groupBy("amount_band").count().collect()
        times.append(time.perf_counter() - t0)
    timings[label] = round(statistics.median(times), 3)
    return timings[label]

t_udf = bench(lambda: tx.withColumn("amount_band", amount_band_udf("amount_mad")), "p3_band_udf")
t_nat = bench(lambda: tx.withColumn("amount_band", amount_band_native(F.col("amount_mad"))), "p3_band_when")
print(f"UDF Python : {t_udf:.2f} s | when/otherwise : {t_nat:.2f} s (mediane de 3 executions)")

tx.withColumn("amount_band", amount_band_native(F.col("amount_mad"))).groupBy("amount_band").count().show()
spark.sql("SELECT amount_band(amount_mad) AS band, count(*) AS n FROM transactions GROUP BY 1").show()

print("--- plan physique avec UDF (noter BatchEvalPython) ---")
tx.withColumn("amount_band", amount_band_udf("amount_mad")).explain()
print("--- plan physique avec when/otherwise (CASE WHEN natif, codegen) ---")
tx.withColumn("amount_band", amount_band_native(F.col("amount_mad"))).explain()

**Pourquoi `when/otherwise` est plus performant** : une UDF Python est une boite noire pour
Catalyst. Chaque ligne doit etre serialisee depuis la JVM vers un processus Python (operateur
`BatchEvalPython`, via Arrow/pickle), evaluee en Python puis renvoyee. `when/otherwise` est
traduit en `CASE WHEN` natif : il reste dans la JVM, beneficie du *whole-stage code generation*
de Tungsten (format binaire UnsafeRow, pas de serialisation) et peut etre optimise par Catalyst
(constant folding, predicate pushdown...).

## 5. Ecriture de la couche Silver (Parquet partitionne par `city`)
Nettoyage : schema type, suppression des lignes invalides, dedoublonnage `(tx_id, status)`,
ajout de `amount_band`, `date`, et des attributs commercant pour les paiements P2M.

In [ ]:
silver = (tx
          .filter(F.col("tx_id").isNotNull() & F.col("event_time").isNotNull() & (F.col("amount_mad") > 0))
          .dropDuplicates(["tx_id", "status"])
          .withColumn("amount_band", amount_band_native(F.col("amount_mad")))
          .withColumn("date", F.to_date("event_time"))
          .join(F.broadcast(merchants.select(F.col("merchant_id").alias("payee_id"),
                                             F.col("name").alias("merchant_name"),
                                             F.col("category").alias("merchant_category"))),
                on="payee_id", how="left"))

with timer("p3_write_silver", timings):
    (silver.repartition("city")
           .write.mode("overwrite")
           .partitionBy("city")
           .parquet(Paths.silver_history))

silver_back = spark.read.parquet(Paths.silver_history)
print("lignes Silver :", silver_back.count())
silver_back.groupBy("city").count().orderBy("city").show()
# Lecture d'une seule partition : partition pruning (PartitionFilters dans le plan)
silver_back.filter(F.col("city") == "Rabat").groupBy("status").count().explain()

## 6. `explain(True)` sur la jointure : plans logique, optimise (Catalyst) et physique

In [ ]:
join_query = (p2m.join(merchants, p2m.payee_id == merchants.merchant_id)
                 .groupBy(merchants.category, merchants.city)
                 .agg(F.sum("amount_mad").alias("montant_total_mad")))
join_query.explain(True)

with open(os.path.join(LOG_DIR, "explain_join_broadcast.txt"), "w") as f:
    f.write(join_query._jdf.queryExecution().toString())

Pour comparaison : on desactive le broadcast automatique, Spark choisit alors un **SortMergeJoin**
(shuffle des deux cotes + tri). Avec `merchants` (300 lignes, quelques Ko, sous le seuil
`spark.sql.autoBroadcastJoinThreshold` = 10 Mo), Catalyst choisit par defaut un
**BroadcastHashJoin** : la petite table est envoyee a chaque executor, la grande n'est pas shufflee.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
with timer("p3_join_sortmerge", timings):
    join_query_smj = (p2m.join(merchants, p2m.payee_id == merchants.merchant_id)
                         .groupBy(merchants.category, merchants.city)
                         .agg(F.sum("amount_mad").alias("montant_total_mad")))
    join_query_smj.collect()
join_query_smj.explain()
with open(os.path.join(LOG_DIR, "explain_join_sortmerge.txt"), "w") as f:
    f.write(join_query_smj._jdf.queryExecution().toString())
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", str(10 * 1024 * 1024))
with timer("p3_join_broadcast", timings):
    join_query.collect()

## 7. Comparaison RDD / DataFrame / Dataset (temps mesures)

In [ ]:
try:
    with open(os.path.join(LOG_DIR, "timings_part2_rdd.json")) as f:
        t2 = json.load(f)
except FileNotFoundError:
    t2 = {}
rows = [
    ("Lecture + comptage",        t2.get("p2_count_events"),   timings.get("p3_count_events")),
    ("Montant total par ville",   t2.get("p2_total_by_city"),  timings.get("p3_total_by_city_df")),
    ("Total par ville (SQL)",     None,                        timings.get("p3_total_by_city_sql")),
    ("Total + moyenne par ville", t2.get("p2_avg_by_city"),    timings.get("p3_total_by_city_df")),
    ("DAG complet sans cache",    t2.get("p2_dag_job_nocache"), timings.get("p3_total_by_city_df")),
]
print(f"{'Traitement':<28}{'RDD (s)':>10}{'DataFrame (s)':>15}")
for r in rows:
    print(f"{r[0]:<28}{(r[1] if r[1] is not None else float('nan')):>10.2f}{(r[2] if r[2] is not None else float('nan')):>15.2f}")
with open(os.path.join(LOG_DIR, "timings_part3_df.json"), "w") as f:
    json.dump(timings, f, indent=2)
print(timings)

**A rediger dans le rapport**
* **RDD** : collection distribuee d'objets Python opaques ; aucune connaissance du schema, pas
  d'optimisation automatique ; chaque `lambda` s'execute dans un worker Python (serialisation
  JVM <-> Python). Controle fin mais code plus long.
* **DataFrame** : donnees tabulaires avec schema ; les requetes (API ou SQL) produisent le meme
  plan logique optimise par **Catalyst** (predicate pushdown, projection pruning, choix du join) et
  execute par **Tungsten** (memoire hors tas, format binaire, code genere). En PySpark, tout le
  calcul reste dans la JVM tant qu'on n'utilise pas d'UDF Python.
* **Dataset** : API typee (Scala/Java) qui combine typage statique a la compilation et Catalyst ;
  n'existe pas en Python (un DataFrame est un `Dataset[Row]`). Les lambdas typees d'un Dataset
  (`map(x => ...)`) sont aussi opaques pour Catalyst.
* Nos mesures : le DataFrame est plus rapide que le RDD sur les memes agregations (moins de
  serialisation, lecture JSON vectorisee, agregation codegen), et SQL = DataFrame (meme plan).

In [ ]:
spark.stop()